# 칩위스퍼러 플랫폼 설정 & 사전 작업 & 바이너리 컴파일 및 프로그램

In [ ]:
PLATFORM = 'CWHUSKY'
#PLATFORM = 'CW308_STM32F3'

In [ ]:
%run '../base/My_Setup.ipynb'

# Husky 및 타겟 보드 오류주입 기본 설정 및 assert(샘플수==클록수) 

- 스크립트 검증은 Husky에서 수행하였으나 Pro CW308_STM32F3에서도 동작 가능성을 고려하여 작성함
  - 스크립트 동작 중에 해당 정보에 대한 경고가 뜰 수 있음

In [ ]:
scope.cglitch_setup()   #clock glitch
#scope.vglitch_setup()  #voltage glitch

# 1 샘플 = 1 클럭
scope.clock.adc_src = 'clkgen_x1'
scope.clock.clkgen_src = 'system'
scope.clock.adc_mul = 1
scope.adc.decimate = 1

# 타겟보드 리셋 함수 reset_target(scope)  ->  ../base/Setup_Generic.ipynb  
reset_target(scope)
time.sleep(1)
reset_target(scope)
time.sleep(1)

print(scope.adc)
print(scope.clock)
print(f'scope.clock.adc_src: {scope.clock.adc_src}')
print(f'scope.clock.clkgen_src: {scope.clock.clkgen_src}')
print(f'scope.clock.adc_mul: {scope.clock.adc_mul}')

# Husky 및 타겟 보드 오류주입 실천

- 오류주입 사전 작업: 오류주입 시점 식별을 위한 파형 형태 분석

In [ ]:
# Tip. 예시 암호함수(OTP)는 오직 for문으로 구성되어 있음.
#      MAX_DATA_LEN를 수정하며 scope.adc.samples를 얻는 과정을 반복하여
#      두 변수에 대한 선형 방정식을 얻으면, for문 연산과 기타 연산의 클럭수를 알 수 있음.  

MAX_DATA_LEN = 40 

random.seed(1)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))

# init
reset_target(scope)
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

# 샘플 개수 설정
MAX_TR_LEN = my_setting_num_samples(target, scope)  

expected_ret, trace = my_get_trace(target, scope)

 
# 오류주입 파라미터 offset 범위 (Husky 전용)    //Pro는 퍼센트 단위
#  - 클럭 신호 high 상태 : 0 < (scope.glitch.phase_shift_steps/2)
#  - 클럭 신호 low 상태  : (scope.glitch.phase_shift_steps/2) < scope.glitch.phase_shift_steps
print(f'i_offset(scope.glitch.offset) in range (0, {scope.glitch.phase_shift_steps})')
print(f'i_width(scope.glitch.width) in range (0, {scope.glitch.phase_shift_steps//2})')


print(f'# of samples is : {scope.adc.samples}')
print(f'expected_ret : {expected_ret.hex(" ")} \n') 

%matplotlib inline
plt.plot(trace) 


- 오류주입 파라미터 탐색

In [ ]:
reset_target(scope)
time.sleep(1)
reset_target(scope)
time.sleep(1)

# 0. 글리치 on/off
scope.glitch.arm_timing = 'after_scope'      # after_scope , no_glitch

"""
There are 5 ways that the glitch module can combine the clock with its glitch pulses:
clock_only: Output only the original input clock.
glitch_only: Output only the glitch pulses - do not use the clock.
clock_or: Output is high if either the clock or glitch are high.
clock_xor: Output is high if clock and glitch are different.
enable_only: Output is high for repeat full clock cycles. width and width_fine have no effect, but offset and offset_fine do.

Some of these settings are only useful in certain scenarios:
Clock glitching: 'clock_or' or 'clock_xor
Voltage glitching: 'glitch_only' or 'enable_only'
"""
# 1. 글리치 모듈 설정
scope.glitch.output = 'clock_xor' # 글리치 출력 모드

# 경고 알림 무시: (ChipWhisperer Target WARNING|File SimpleSerial2.py:506) Read timed out 
logging.getLogger('ChipWhisperer Target').setLevel(logging.ERROR)
# 경고 알림 무시: (ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
logging.getLogger('ChipWhisperer Glitch').setLevel(logging.ERROR)


#튜토리얼 클럭 오류주입원 파라미터 예시 (같은 타겟 보드라도 최적 파라미터가 다르며, 연결 케이블 영향도 엄청 큼)
#(예제) Pro_CW308_STM32F3:
#   scope.glitch.offset = -48.828125
#   scope.glitch.width = -10.9375 #루프스킵   ,   -8.984375 #1byte 오류


# 결과 저장 [오동작 행위, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset]
# 오동작 행위:
#   0: ret is None , scope.capture timeout
#   1: nomal
#   2: for loop skip
#   3: one faulty byte 
#   4: few faulty bytes ( <5)
#   5: unknown
cglitch_result = []
# 기타 데이터 ([expected_ret, actual_ret])
cglitch_extra_data = []


for i_ext_offset in trange(170, 183, desc='i_ext_offset', leave=False):

    for i_offset in trange(0, (scope.glitch.phase_shift_steps), 100, desc='i_offset', leave=False):

        for i_width in trange(0, (scope.glitch.phase_shift_steps//2), 100, desc='i_width', leave=False): 
            if i_width == 0:
                continue
            if (i_offset + i_width) > (scope.glitch.phase_shift_steps):
                continue      

            for _ in range(10):
                # 2. 파라미터 설정 (이 부분을 계속 바꾸면서 공격합니다)
                scope.glitch.ext_offset = i_ext_offset  # 신호수집 트리거 시작후 n클럭 이후
                scope.glitch.offset = i_offset          # 글리치 오프셋 (Offset): 1클럭 내에서 클럭 라이징 후 언제 떨어뜨릴지
                scope.glitch.width = i_width            # 글리치 폭 (Width): 1클럭 내에서 전압을 얼마나 오래 떨어뜨릴지
                
                # init  
                reset_target(scope)              
                my_fsr_cmd(target, 0x81, 'k', data_k)
                my_fsr_cmd(target, 0x81, 'p', data_p)
                my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
                
                # 암호 연산 with 오류주입
                target.flush()
                scope.arm()
                target.send_cmd(cmd=0x82, scmd=ord('c'), data=[]) 
                actual_ret = target.read_cmd(timeout=500)   

                # 타겟 보드가 멈춘 경우                 
                if (actual_ret is None) or (scope.capture()):
                    cglitch_result.append([0, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])                    
                    continue            
                
                target.flush()
                target.send_cmd(cmd=0x83, scmd=ord('r'), data=[]) 
                actual_ret = target.read_cmd(timeout=500)
                if (actual_ret is None):
                    cglitch_result.append([0, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])                    
                    continue                         

                # 이하 타겟 보드 정상상태
                actual_ret = actual_ret[3:3 + actual_ret[2]]

                # 오류주입이 타겟보드에 아무 영향 없는 경우                
                if (expected_ret == actual_ret):
                    cglitch_result.append([1, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                
                # for loop 스킵 성공
                elif (scope.adc.trig_count < (scope.adc.samples-300)):
                    cglitch_result.append([2, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\nfor loop skip ')    
                    print(f'i_ext_offset : {scope.glitch.ext_offset} ')                            
                    print(f'i_width      : {scope.glitch.width} ')       
                    print(f'i_offset     : {scope.glitch.offset} ')      
                    print(f'expected_ret : {expected_ret.hex(" ")} ')   
                    print(f'actual_ret   : {actual_ret.hex(" ")} ')  , 1600, 1800, 1900
                    plt.plot(np.array(scope.get_last_trace()))

                # 1 바이트 오류주입 성공
                elif (sum(g != r for g, r in zip(expected_ret, actual_ret)) == 1):                    
                    cglitch_result.append([3, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\none faulty byte ')    
                    print(f'i_ext_offset : {scope.glitch.ext_offset} ')                            
                    print(f'i_width      : {scope.glitch.width} ')       
                    print(f'i_offset     : {scope.glitch.offset} ')      
                    print(f'expected_ret : {expected_ret.hex(" ")} ')   
                    print(f'actual_ret   : {actual_ret.hex(" ")} ')  

                # 적은 바이트 오류주입 성공                
                elif (sum(g != r for g, r in zip(expected_ret, actual_ret)) < 5):                    
                    cglitch_result.append([4, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    print(f'\nfew faulty bytes ( <5) ')   
                    print(f'i_ext_offset : {scope.glitch.ext_offset} ')                            
                    print(f'i_width      : {scope.glitch.width} ')       
                    print(f'i_offset     : {scope.glitch.offset} ')      
                    print(f'expected_ret : {expected_ret.hex(" ")} ')   
                    print(f'actual_ret   : {actual_ret.hex(" ")} ')  

                else:
                    cglitch_result.append([5, scope.glitch.ext_offset, scope.glitch.width, scope.glitch.offset])   
                    cglitch_extra_data.append([expected_ret, actual_ret])   

 
print('Done! \n')

- 오류주입 파라미터 통계 분석

In [ ]:
cglitch_result = np.array(cglitch_result, dtype=np.float64) 

print(f'\n# of for loop skip is {sum(cglitch_result[:,0] == 2)}')
print(f'best of i_offset: {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 2,3], keepdims=False).mode} ') 
print(f'best of i_width : {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 2,2], keepdims=False).mode} ')  

print(f'\n# of one faulty byte is {sum(cglitch_result[:,0] == 3)}')     
print(f'best of i_offset: {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 3,3], keepdims=False).mode}') 
print(f'best of i_width : {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 3,2], keepdims=False).mode}')  

print(f'\n# of few faulty bytes ( <5) is {sum(cglitch_result[:,0] == 4)}')     
print(f'best of i_offset: {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 4,3], keepdims=False).mode}') 
print(f'best of i_width : {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 4,2], keepdims=False).mode}')  

print(f'\n# of etc. is {sum(cglitch_result[:,0] == 5)}')      
print(f'best of i_offset: {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 5,3], keepdims=False).mode}') 
print(f'best of i_width : {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 5,2], keepdims=False).mode}') 

print(f'\n# of freezing is {sum(cglitch_result[:,0] == 0)}')      
print(f'best of i_offset: {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 0,3], keepdims=False).mode}') 
print(f'best of i_width : {sp.stats.mode(cglitch_result[cglitch_result[:,0] == 0,2], keepdims=False).mode}') 

print(f'\n# of nomal is {sum(cglitch_result[:,0] == 1)}')

In [ ]:
scope.dis()
target.dis()